# DPIRD weather data ingestion
Port of Mapping/DPIRD/preprocess/preprocess.py 

In [0]:
from pathlib import Path
import os, yaml, gzip, tarfile, shutil, pytz
import pandas as pd
import numpy as np
import xarray as xr
from concurrent.futures import ThreadPoolExecutor

: 

## Define paths

In [0]:
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)
    
config= load_yaml('configs/config.yaml')
DPIRD_path_config= config['DPIRD_data_path']
raw_tar_path= Path(DPIRD_path_config['raw-file'])
untar_folder= Path(DPIRD_path_config['untar-folder'])

## Untar actions

In [ ]:
from collections import defaultdict
"""
Unzip DPIRD station data by organizing into YYYY/MM/station.csv files. 
Extracts to a temp DPIRD_staging directory, unzips concurrently in place and cleans up all .csv.gz
"""
if not raw_tar_path.exists():
    raise Exception(f"Check config.yaml: {raw_tar_path} does not exist.")

# 1. Setup temp staging path 
tmp_stage_dir = Path("DPIRD_staging")
if tmp_stage_dir.exists():
    shutil.rmtree(tmp_stage_dir) 
tmp_stage_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting initial tarball to temporary node storage at {tmp_stage_dir}...")
with tarfile.open(raw_tar_path, "r:gz") as tar:
    tar.extractall(path=tmp_stage_dir)

def is_gzip_file(file_path):
    try:
        with open(file_path, 'rb') as f:
            return f.read(2) == b'\x1f\x8b'
    except Exception:
        return False

""" 
Check gzip type, unzip IN-PLACE in the staging dir, and remove old .gz
"""  
def process_station_file(source_file_path):
    source_file = Path(source_file_path)
    yyyymm = source_file.parent.name
    if len(yyyymm) != 6 or not yyyymm.isdigit():
        return  
    
    # Map to YYYY/MM relative to the staging root
    year, month = yyyymm[:4], yyyymm[4:]
    dest_dir = source_file.parent.parent / year / month
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    # Clean the output extension
    station_name = source_file.name.replace('.csv.gz', '.csv').replace('.gz', '.csv')
    
    # Requested error logging
    if not station_name.endswith('.csv'):
        station_name += '.csv'
        print(f"Warning: Unexpected file type found. Folder: {yyyymm}, File: {source_file.name}")
        
    dest_file_path = dest_dir / station_name

    # Check magic bytes and decompress in-place
    if source_file.is_file() and is_gzip_file(source_file):
        with gzip.open(source_file, 'rb') as f_in, open(dest_file_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
            
        # Delete the .gz file after untar
        source_file.unlink()

    return f"{year}/{month}"

# Gather unzipped folders in the temporary directory
all_dirs = [d for d in tmp_stage_dir.rglob('*') if d.is_dir() and len(d.name) == 6 and d.name.isdigit()]
target_files = []
for folder in all_dirs:
    target_files.extend(list(folder.glob('*.*')))

#Build dict to store {YYYY: [MM]} key-val pair
year_month_dict = defaultdict(set)
if target_files:
    print(f"Processing {len(target_files)} files using ThreadPoolExecutor...")
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        results= list(executor.map(process_station_file, target_files))

        
        for res in results:
            if res: # ignore Nones
                y, m = res.split('/')
                year_month_dict[y].add(m)

year_month_dict = {k: sorted(list(v)) for k, v in year_month_dict.items()}

# Clean up the legacy YYYYMM folders now that all threads are done
def remove_legacy_dir(folder):
    if folder.exists():
        shutil.rmtree(folder, ignore_errors=True)

print("Cleaning up legacy YYYYMM directories concurrently...")
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    list(executor.map(remove_legacy_dir, all_dirs))

for folder in all_dirs:
    try:
        p = folder.parent
        if p.exists() and not any(p.iterdir()) and p != tmp_stage_dir:
            p.rmdir()
    except Exception:
        pass

print(f"In-place extraction complete")
print(f"Generated Dictionary: {year_month_dict}")
print(f"Files are staged at and ready for downstream tasks at: {tmp_stage_dir}")